# 08 — R0~R3 비교를 효용 이득 $G_r$로 다시 계산

R0~R3 비교 실험에서 학습한 모델과 저장된 점수를 **그대로** 쓰고, 비교 지표만 새 효용 계산식으로 바꿉니다. 재학습·GPU·새 LLM 호출은 없습니다. Colab CPU 런타임이나 로컬에서 돌릴 수 있습니다.

## 식

$$G_r = A_r - H(C_r)$$

- $A_r$: 라우터 $r$이 고른 단계가 **첫 시도에 맞힌 비율**. 점수 파일의 `success_{low,medium,high}`(0/1)를 씁니다. 연속값 `quality`는 쓰지 않습니다.
- $C_r$: 평균 비용. 단계 가격 low : medium : high = 1 : 3 : 5, high = 1로 맞춰 0.2 / 0.6 / 1.0입니다.
- $H(C_r)$: 비용 $C_r$ **이하**로 고정 단계(모두 low · 모두 medium · 모두 high)와 그 혼합이 낼 수 있는 가장 좋은 첫 시도 해결률. 세 점 $(c_t, A_t)$를 **위로 볼록한 껍질**로 잇습니다. medium 점이 low–high 직선보다 아래에 있으면 medium을 건너뛰고 low와 high를 바로 잇습니다. 이렇게 해야 기준선이 과소평가되어 $G$가 부풀지 않습니다.
  - 비싼 단계가 오히려 못한 원천도 있습니다(예: xRouteBench는 high가 medium보다 낮을 때가 있음). 그 경우 기준선은 내려가지 않고 가장 좋은 점에서 평평하게 유지됩니다. 더 싸게 더 잘 푸는 방법이 있는데 돈을 더 쓴 라우터에 이득을 주지 않기 위해서입니다. 라우터 비교 실험(EV-10) 노트북 `router_final/11_gain`과 같은 정의입니다.

$G_r > 0$이면 같은 돈으로 무작위로 섞는 것보다 첫 시도에 더 많이 맞혔다는 뜻입니다.

## 대상 (모델은 기존 그대로)

| 실험 | 라우터 | 튜닝 데이터 | 평가 데이터 |
|---|---|---|---|
| A (RT-05) | R0(CE), R1(성공 BCE), R2(pairwise) | D_dev | 봉인 test (RouterBench / xRouteBench) |
| G1 (RT-06) | R1-natural·balanced, R3-natural·balanced | calibration | selection, task holdout |

## 운영점

- **λ=0:** 모델 출력 그대로 `argmax(score)`. 조절값이 없는 상태입니다.
- **G 최대:** 결정 규칙 `argmax(score − λ × 단계 가격)`의 λ를 0과 $10^{-4}$~$10^{2}$ 로그 간격 601점에서 훑어, **튜닝 데이터에서 $G$가 가장 큰 λ**를 원천마다 고르고 평가 데이터에 그대로 적용합니다. λ는 곡선을 따라 움직이는 손잡이로만 쓰고, 고르는 기준은 $G$ 자체라 비용–해결률 환율을 정하지 않습니다. 동률이면 비용이 작은 쪽, 그다음 λ가 작은 쪽입니다.

점수 변환은 기존과 같습니다(R0 softmax, R1 sigmoid, R2·R3 raw logit). 기존 [λ 정밀 탐색](../../../experiments/results/lambda-sweep-20261002/REPORT.md)과 같은 점수 파일·같은 튜닝/평가 분리를 쓰고, 달라진 것은 ① 정답 기준(quality → 첫 시도 success), ② 비용(데이터 실제 비용 → 1:3:5), ③ 고르는 기준(예산·유지율 → $G$)입니다.

## 집계와 불확실성

- 원천별로 계산한 뒤 두 원천을 같은 비중으로 평균합니다(source-macro).
- 문항 bootstrap 2,000회로 95% 구간을 냅니다. 같은 실험·같은 데이터의 라우터들은 같은 재표집을 공유하므로, 라우터 간 차이 $\Delta G$의 구간도 짝지어 계산합니다.
- 참고 상한으로 oracle(맞히는 가장 싼 단계, 아무도 못 맞히면 low)의 $G$도 냅니다.
- 단일 seed 모델입니다. 구간은 문항 표본의 흔들림만 반영하고 학습 seed의 흔들림은 반영하지 않습니다.

In [ ]:
# 환경 설정. Colab이면 Drive를 붙이고, 로컬이면 환경 변수 또는 기본 경로를 씁니다.
import hashlib, json, os, shutil, subprocess, tarfile, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

try:
    display
except NameError:  # 노트북 밖에서 실행할 때
    display = print

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/LLM_Router')
    A_ROOT = Path('/content/gain-inputs/A')                       # tar를 풀어 둘 임시 위치 (Drive에 복사하지 않음)
    A_TAR_DIR = DRIVE / 'routing-objectives-continual-v1-A-20260924'
    G1_ROOT = DRIVE / 'routing-generalization-v1-20260925'      # merged-runs/, holdout-scores/ 가 있는 폴더
    OUT_DIR = DRIVE / 'gain-metric-r0r3-20261021'
else:
    REPO = Path(os.environ.get('LLM_ROUTER_REPO', Path.cwd())).resolve()
    while not (REPO / 'ml').is_dir() and REPO != REPO.parent:
        REPO = REPO.parent
    A_ROOT = Path(os.environ.get('GAIN_A_ROOT', Path.home() / '.cache/llm-router-gain-inputs/A'))
    A_TAR_DIR = Path(os.environ.get('GAIN_A_TAR_DIR', REPO / '.gain-inputs/tars'))
    G1_ROOT = Path(os.environ.get('GAIN_G1_ROOT', REPO / 'experiments/results/routing-generalization-v1-20260925'))
    OUT_DIR = Path(os.environ.get('GAIN_OUT_DIR', REPO / 'experiments/results/gain-metric-r0r3-20261021'))

# 로컬 확인용: 일부 입력만 있을 때 빠진 라우터를 건너뜁니다. 본 실행에서는 False로 둡니다.
ALLOW_MISSING = os.environ.get('GAIN_ALLOW_MISSING', '0') == '1'
print('Colab' if IN_COLAB else 'local', '| A:', A_ROOT, '| G1:', G1_ROOT, '| out:', OUT_DIR)

## 입력

A 점수는 두 tar(`comparison-results.tar`, `sealed-inference.tar`)에, G1 점수는 `routing-generalization-v1-20260925`에 있습니다. 모든 파일은 λ 정밀 탐색 manifest에 기록된 SHA-256과 맞아야 합니다. 맞지 않으면 멈춥니다.

A tar는 Drive `LLM_Router/routing-objectives-continual-v1-A-20260924/`에 있는 사본을 먼저 쓰고(같은 SHA 확인됨), 없으면 GCS에서 받습니다(`gcloud storage cp`, 읽기 전용). tar는 임시 폴더에 풀고 필요한 파일만 꺼냅니다.

In [ ]:
GCS_A = {
    'validation': 'gs://llm-router-mlserver/training-runs/objectives-continual-v1-A-20260924/comparison-results.tar',
    'test': 'gs://llm-router-mlserver/training-runs/objectives-continual-v1-A-20260924/sealed-tests-v1/sealed-inference.tar',
}
LOCAL_A = {'validation': A_TAR_DIR / 'comparison-results.tar', 'test': A_TAR_DIR / 'sealed-tests/sealed-inference.tar'}
SOURCES = ('routerbench', 'xroutebench')

# (실험, 라우터, 점수 방식, 튜닝 split, {split: [(상대 경로, sha256), ...]})
RUNS = [
    ('A', 'R0', 'ordinal', 'D_dev', {
        'D_dev': [(A_ROOT / 'ordinal/validation-scores.parquet', 'c7cde5f2f69336f6c0b4f9faefcd35cbf1bd56de1b6f79d8bcea8779799ea776')],
        'test': [(A_ROOT / 'ordinal/test_routerbench-scores.parquet', 'fafecd0b0892e79d866d5b6a771af25827e6b93708816bb76a612bb458f1a937'),
                 (A_ROOT / 'ordinal/test_xroutebench-scores.parquet', 'd28ed3101cc6665511f895fb336be2ed57b2fb383f78bee70566908dbcfc4a70')]}),
    ('A', 'R1', 'pointwise', 'D_dev', {
        'D_dev': [(A_ROOT / 'pointwise/validation-scores.parquet', '0edfe7f64ba8f96bfb6d4022406bf4d7b714b155e819d69c7fa006f32af1635b')],
        'test': [(A_ROOT / 'pointwise/test_routerbench-scores.parquet', '2b0ed365810eae187823d8ed6c96f74aea383f6c089f5bd8e3a6b94827693719'),
                 (A_ROOT / 'pointwise/test_xroutebench-scores.parquet', 'c9e292a3cb4d266bf4da808be679ff72092196cdeacd040511833ffd4751dabc')]}),
    ('A', 'R2', 'pairwise', 'D_dev', {
        'D_dev': [(A_ROOT / 'pairwise/validation-scores.parquet', '97942b36eacf4fd9c3fdcf49238facd010e577eef3c3f19e17f858760cfac597')],
        'test': [(A_ROOT / 'pairwise/test_routerbench-scores.parquet', '69f9698388a975508469699f9d26b493db814f2d92e42afde0175e2a23a25e83'),
                 (A_ROOT / 'pairwise/test_xroutebench-scores.parquet', '35e27c12706c162a4c276d28586679aa2cef65db292b7c4ccc06c784c277babe')]}),
]
G1_SHA = {
    'r1-natural': ('0f1a967d2a2710b88dd048cffbc02890e18a77d414a4d722f4ad98607497563f', 'd7273a53bdbed7e27d0789c4aa29ef4e7fdf0ab0af5d5913412dd1bccb21d001', '83a815cbd295382d9a4b747e04a20baa298a88c955e104368ae172d642aacd2e'),
    'r1-balanced': ('d94f0c78b8066d76a64810b301c55469a45b06af552c74d0b36d9aa56bb8b16b', 'ba4c270e9a7bfd340ba1751abecb16b5d670bafe0255cf70d6fe4d1e68a8f52e', '6eac567c9ad85bc4180f7844f90ad89b5e4e536a6f2a1e96780db525ea098133'),
    'r3-natural': ('40132d80a888a2050f4d0e6d89edb24975cb557b1e007cc9979cb6048d2e087e', '7e8aaeeb01cc03c579ad3115a5b173ed47058dd14f42197e1d09856604b683b1', '122c78a5fdf9d14120f7667943e8252f3b4ac209b21c36bb177aa80a7bae67dc'),
    'r3-balanced': ('3fe09c24d3f813c0098d5ba7ec9f86fc0f66c22c4b3b5b610f3be5906ed88ca0', '0ec6de322e9594ea16709946d2f6154dd9da964a33d86e4dfc788d0661b43163', 'bdbccbb4a0bf4765a4f89b31e1b48f8890176e41d6b7e14142822180fc0ac1fb'),
}
for arm, (cal, sel, hold) in G1_SHA.items():
    RUNS.append(('G1', arm.upper(), 'pointwise' if arm.startswith('r1') else 'regret', 'calibration', {
        'calibration': [(G1_ROOT / f'merged-runs/{arm}/calibration-scores.parquet', cal)],
        'selection': [(G1_ROOT / f'merged-runs/{arm}/selection-scores.parquet', sel)],
        'task_holdout': [(G1_ROOT / f'holdout-scores/{arm}-scores.parquet', hold)],
    }))


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def fetch_a_scores():
    # tar 두 개(Drive 사본, 없으면 GCS)를 임시 폴더에 풀고, method/파일 이름이 맞는 parquet만 A_ROOT로 옮긴다.
    wanted = {p for exp, _, _, _, splits in RUNS if exp == 'A' for files in splits.values() for p, _ in files}
    if all(p.is_file() for p in wanted):
        return
    if IN_COLAB and not all(p.is_file() for p in LOCAL_A.values()):
        from google.colab import auth
        auth.authenticate_user()
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        for name, uri in GCS_A.items():
            tar_path = tmp / f'{name}.tar'
            if LOCAL_A[name].is_file():
                shutil.copy(LOCAL_A[name], tar_path)
            else:
                subprocess.run(['gcloud', 'storage', 'cp', uri, str(tar_path)], check=True)
            with tarfile.open(tar_path) as tar:
                try:
                    tar.extractall(tmp / name, filter='data')
                except TypeError:  # Python < 3.11.4
                    tar.extractall(tmp / name)
            tar_path.unlink()
        for target in wanted:
            method, fname = target.parent.name, target.name
            hits = [p for p in tmp.rglob(fname) if p.parent.name == method]
            if len(hits) != 1:
                raise RuntimeError(f'{method}/{fname}: tar 안에서 {len(hits)}개를 찾았습니다')
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(hits[0]), target)


try:
    fetch_a_scores()
except Exception as exc:  # 로컬 확인에서 GCS 인증이 없을 때
    if not ALLOW_MISSING:
        raise
    print('A 점수를 받지 못했습니다(ALLOW_MISSING):', exc)

available, missing, input_sha = [], [], {}
for run in RUNS:
    files = [(p, s) for split_files in run[4].values() for p, s in split_files]
    absent = [str(p) for p, _ in files if not p.is_file()]
    if absent:
        missing.append((run[0], run[1], absent))
        continue
    for p, expected in files:
        got = sha256(p)
        if got != expected:
            raise RuntimeError(f'SHA-256 불일치: {p}\n  기대 {expected}\n  실제 {got}')
        input_sha[str(p)] = got
    available.append(run)

for exp, arm, absent in missing:
    print(f'[빠짐] {exp} {arm}:', *absent, sep='\n  ')
if missing and not ALLOW_MISSING:
    raise RuntimeError('입력 파일이 빠졌습니다. 위 경로를 채우거나 G1_ROOT/A_ROOT를 고치세요.')
print('사용할 라우터:', [f'{r[0]}:{r[1]}' for r in available])

## 계산 함수

In [ ]:
TIERS = ('low', 'medium', 'high')
PRICE = np.array([0.2, 0.6, 1.0])          # 1:3:5, high = 1
LAMBDAS = np.concatenate([[0.0], np.logspace(-4, 2, 601)])
N_BOOT = 2000
SEED = 20261021


def transform(method, raw):
    if method == 'pointwise':
        return 1.0 / (1.0 + np.exp(-np.clip(raw, -30.0, 30.0)))
    if method == 'ordinal':
        e = np.exp(raw - raw.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)
    return raw


def load(paths):
    frame = pd.concat([pd.read_parquet(p) for p, _ in paths], ignore_index=True)
    # 같은 실험·split의 라우터끼리 같은 행 순서를 갖게 해 bootstrap 재표집을 짝짓는다.
    return frame.sort_values('sample_id', kind='stable').reset_index(drop=True)


def matrices(frame, method):
    get = lambda p: frame[[f'{p}_{t}' for t in TIERS]].to_numpy(float)  # noqa: E731
    s = get('success')
    if np.isnan(s).any() or not np.isin(s, (0.0, 1.0)).all():
        raise ValueError('success 열은 0/1이어야 합니다')
    return {'score': transform(method, get('score')), 's': s, 'ids': frame['sample_id'].to_numpy()}


def hull(fixed_acc):
    # 고정 단계 세 점의 위로 볼록한 껍질 꼭짓점 (비용, 해결률).
    a_low, a_med, a_high = fixed_acc
    chord = a_low + (a_high - a_low) * (PRICE[1] - PRICE[0]) / (PRICE[2] - PRICE[0])
    keep = [0, 1, 2] if a_med > chord else [0, 2]
    return PRICE[keep], np.asarray(fixed_acc)[keep]


def baseline(cost, fixed_acc):
    # 비용 cost 이하에서 고정 단계 혼합이 낼 수 있는 최고 해결률: 껍질의 최고점 뒤는 평평하다.
    xs, ys = hull(fixed_acc)
    top = int(np.argmax(ys))
    return np.interp(cost, xs[:top + 1], ys[:top + 1])


def gain(choice, s):
    # choice: 문항별 단계 index. 반환: A, C, H, G, 단계 비율.
    rows = np.arange(len(s))
    acc = s[rows, choice].mean()
    cost = PRICE[choice].mean()
    h = baseline(cost, s.mean(axis=0))
    rates = np.bincount(choice, minlength=3) / len(choice)
    return {'A': acc, 'C': cost, 'H': h, 'G': acc - h,
            'low_rate': rates[0], 'medium_rate': rates[1], 'high_rate': rates[2]}


def choices(score, lam):
    return np.argmax(score - lam * PRICE, axis=1)


def oracle_choice(s):
    # 맞히는 가장 싼 단계, 아무도 못 맞히면 low.
    first = np.argmax(s > 0, axis=1)
    return np.where(s.any(axis=1), first, 0)


def curve(m):
    return pd.DataFrame([{'lambda': lam, **gain(choices(m['score'], lam), m['s'])} for lam in LAMBDAS])


def pick_gmax(tune_curve):
    best = tune_curve.sort_values(['G', 'C', 'lambda'], ascending=[False, True, True]).iloc[0]
    return float(best['lambda'])


def boot_index(n, source):
    # 같은 (원천, 표본 수)이면 같은 재표집 → 라우터 간 짝지은 비교.
    rng = np.random.default_rng([SEED, SOURCES.index(source), n])
    return rng.integers(0, n, size=(N_BOOT, n))


def boot_gain(choice, s, idx):
    picked = s[np.arange(len(s)), choice]
    acc = picked[idx].mean(axis=1)
    cost = PRICE[choice][idx].mean(axis=1)
    fixed = s[idx].mean(axis=1)                      # (B, 3)
    h = np.array([baseline(c, f) for c, f in zip(cost, fixed)])
    return acc - h

## 실행

In [ ]:
curves, points, boots = [], [], {}
for exp, arm, method, tune_split, splits in available:
    frames = {name: load(files) for name, files in splits.items()}
    for source in SOURCES:
        per = {name: matrices(f[f.source_dataset.eq(source)].reset_index(drop=True), method)
               for name, f in frames.items()}
        tune_curve = curve(per[tune_split])
        chosen = {'lambda0': 0.0, 'gmax': pick_gmax(tune_curve)}
        for name, m in per.items():
            role = 'tune' if name == tune_split else 'eval'
            c = tune_curve if role == 'tune' else curve(m)
            curves.append(c.assign(experiment=exp, arm=arm, source=source, split=name, role=role))
            fixed = m['s'].mean(axis=0)
            hx, _ = hull(fixed)
            base = {'experiment': exp, 'arm': arm, 'source': source, 'split': name, 'role': role, 'n': len(m['s']),
                    **{f'fixed_{t}': fixed[i] for i, t in enumerate(TIERS)},
                    'medium_on_hull': bool(len(hx) == 3)}
            idx = boot_index(len(m['s']), source) if role == 'eval' else None
            for criterion, lam in chosen.items():
                ch = choices(m['score'], lam)
                row = {**base, 'criterion': criterion, 'lambda': lam, **gain(ch, m['s'])}
                if idx is not None:
                    b = boot_gain(ch, m['s'], idx)
                    boots[(exp, arm, source, name, criterion)] = b
                    row['G_lo'], row['G_hi'] = np.quantile(b, [0.025, 0.975])
                points.append(row)
            if not any(p['arm'] == 'ORACLE' and (p['experiment'], p['source'], p['split']) == (exp, source, name) for p in points):
                points.append({**base, 'arm': 'ORACLE', 'criterion': 'oracle', 'lambda': np.nan,
                               **gain(oracle_choice(m['s']), m['s'])})

curves = pd.concat(curves, ignore_index=True)
points = pd.DataFrame(points)
print(points.shape, curves.shape)

In [ ]:
# source-macro 요약: 두 원천 평균. 구간은 원천별 bootstrap 값을 같은 회차끼리 평균해 낸다.
def macro(points, boots):
    ev = points[points.role.eq('eval')]
    rows = []
    for (exp, arm, split, criterion), g in ev.groupby(['experiment', 'arm', 'split', 'criterion'], sort=False):
        if g.source.nunique() != len(SOURCES):
            continue
        row = {'experiment': exp, 'arm': arm, 'split': split, 'criterion': criterion}
        for col in ('A', 'C', 'H', 'G', 'low_rate', 'medium_rate', 'high_rate'):
            row[col] = g[col].mean()
        keys = [(exp, arm, s, split, criterion) for s in SOURCES]
        if all(k in boots for k in keys):
            b = np.mean([boots[k] for k in keys], axis=0)
            row['G_lo'], row['G_hi'] = np.quantile(b, [0.025, 0.975])
        row.update({f'lambda_{s}': v for s, v in zip(g.source, g['lambda'])})
        rows.append(row)
    return pd.DataFrame(rows)


def paired(boots):
    # 같은 실험·split·기준에서 라우터 쌍의 ΔG(행 − 열), source-macro.
    rows = []
    keys = {(e, sp, cr) for e, _, _, sp, cr in boots}
    for exp, split, criterion in sorted(keys):
        arms = sorted({a for e, a, _, sp, cr in boots if (e, sp, cr) == (exp, split, criterion)})
        for i, a in enumerate(arms):
            for b in arms[i + 1:]:
                da = np.mean([boots[(exp, a, s, split, criterion)] for s in SOURCES], axis=0)
                db = np.mean([boots[(exp, b, s, split, criterion)] for s in SOURCES], axis=0)
                d = da - db
                lo, hi = np.quantile(d, [0.025, 0.975])
                rows.append({'experiment': exp, 'split': split, 'criterion': criterion, 'a': a, 'b': b,
                             'dG_mean_boot': d.mean(), 'dG_lo': lo, 'dG_hi': hi,
                             'separated': bool(lo > 0 or hi < 0)})
    return pd.DataFrame(rows)


summary = macro(points, boots)
pairs = paired(boots)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.width', 200)
display(summary.drop(columns=[c for c in summary if c.startswith('lambda_')]))

### 원천별 값과 고정 단계 기준선

`medium_on_hull = False`이면 medium 점이 low–high 직선 아래라 기준선에서 빠졌다는 뜻입니다.

In [ ]:
cols = ['experiment', 'arm', 'source', 'split', 'criterion', 'lambda', 'n', 'A', 'C', 'H', 'G', 'G_lo', 'G_hi',
        'low_rate', 'medium_rate', 'high_rate', 'fixed_low', 'fixed_medium', 'fixed_high', 'medium_on_hull']
display(points[points.role.eq('eval')][cols])

### 라우터 간 차이 (짝지은 bootstrap)

`separated = True`는 95% 구간이 0을 포함하지 않는다는 뜻입니다. 단일 seed라 학습 흔들림은 빠져 있습니다.

In [ ]:
display(pairs)

## 그림

곡선은 λ를 훑으며 얻은 (비용, 첫 시도 해결률)이고, 회색 선은 고정 단계 기준선 $H$입니다. 곡선이 회색 선 위에 있는 만큼이 $G$입니다. 회색 선은 비용 이하에서 고를 수 있는 최선이라 최고점 뒤로는 평평합니다.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
for (exp, split), g in curves[curves.role.eq('eval')].groupby(['experiment', 'split'], sort=False):
    fig, axes = plt.subplots(1, len(SOURCES), figsize=(5.5 * len(SOURCES), 4.2), squeeze=False)
    for ax, source in zip(axes[0], SOURCES):
        gs = g[g.source.eq(source)]
        if gs.empty:
            continue
        ref = points[(points.experiment.eq(exp)) & (points.split.eq(split)) & (points.source.eq(source))].iloc[0]
        fixed = np.array([ref.fixed_low, ref.fixed_medium, ref.fixed_high])
        xs = np.linspace(PRICE[0], PRICE[2], 100)
        ax.plot(xs, baseline(xs, fixed), color='0.6', lw=1.5, label='H (fixed-tier mix)')
        ax.scatter(PRICE, fixed, color='0.4', zorder=3, s=18)
        for arm, ga in gs.groupby('arm', sort=False):
            ga = ga.drop_duplicates(['C', 'A'])
            ax.plot(ga.C, ga.A, lw=1.4, label=arm)
            p = points[(points.experiment.eq(exp)) & (points.arm.eq(arm)) & (points.split.eq(split))
                       & (points.source.eq(source)) & (points.criterion.eq('gmax'))]
            if len(p):
                ax.scatter(p.C, p.A, marker='*', s=90, zorder=4)
        ax.set_title(f'{exp} · {split} · {source}')
        ax.set_xlabel('mean cost C (1:3:5, high=1)')
        ax.set_ylabel('first-attempt success A')
        ax.set_xlim(0.15, 1.05)
        ax.grid(alpha=0.3)
    axes[0][0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{exp.lower()}-{split}-gain.png', dpi=150)
    plt.show()

## 저장

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
curves.to_csv(OUT_DIR / 'curves.csv', index=False)
points.to_csv(OUT_DIR / 'points.csv', index=False)
summary.to_csv(OUT_DIR / 'macro-summary.csv', index=False)
pairs.to_csv(OUT_DIR / 'paired-differences.csv', index=False)
(OUT_DIR / 'manifest.json').write_text(json.dumps({
    'metric': 'G = A - H(C); A = first-attempt success of chosen tier; C = mean tier price 1:3:5 (high=1); '
              'H = upper concave hull of fixed-tier points',
    'tier_price': PRICE.tolist(),
    'decision_rule': 'argmax(score - lambda * tier_price)',
    'lambda_grid': {'zero': True, 'logspace_min': 1e-4, 'logspace_max': 100, 'points': len(LAMBDAS)},
    'criteria': {'lambda0': 'model output as is', 'gmax': 'per source, max G on tuning split; ties -> lower C, lower lambda'},
    'tuning_split': {'A': 'D_dev', 'G1': 'calibration'},
    'bootstrap': {'resamples': N_BOOT, 'seed': SEED, 'paired_across_arms': True},
    'routers_used': [f'{r[0]}:{r[1]}' for r in available],
    'routers_missing': [f'{e}:{a}' for e, a, _ in missing],
    'input_sha256': {str(Path(k).relative_to(A_ROOT.parent)) if Path(k).is_relative_to(A_ROOT.parent) else str(Path(k).relative_to(G1_ROOT.parent)) if Path(k).is_relative_to(G1_ROOT.parent) else k: v for k, v in input_sha.items()},
    'retraining': False,
}, indent=2, ensure_ascii=False))
print('저장:', OUT_DIR)